# 05 — Gold: Métricas Analíticas
## Módulo 3 | Camada Gold | Seção do Exame: S3 (22%)

**O que este notebook faz:**
- Cria `gold.daily_summary` — resumo diário por cidade (avg/min/max temp, umidade, vento)
- Cria `gold.city_ranking_temp` — ranking de cidades por temperatura atual
- Cria `gold.weather_alerts` — view com alertas: HIGH_TEMP, RAIN_RISK, STORM_RISK
- Cria `gold.temperature_moving_avg` (Desafio 3) — média móvel de temperatura 24h/72h via window functions
- Aplica `OPTIMIZE` + `ZORDER` nas tabelas Gold

**Princípio Gold:** Dados prontos para consumo direto por BI, dashboards e ML.  
Toda lógica de negócio fica aqui — nenhuma transformação nos dashboards.

**Conceito do exame (S3):** Diferença entre `TABLE`, `VIEW` e `MATERIALIZED VIEW`:  
- `TABLE` Delta = dados físicos persistidos, alta performance de leitura  
- `VIEW` = consulta lógica, sempre atual, sem armazenamento físico  
- `MATERIALIZED VIEW` (Lakeflow) = resultado pré-computado e atualizado automaticamente

---

### ✅ Checklist deste notebook
- [ ] Célula 1 — `daily_summary` criada com `avg_temp`, `min_temp`, `max_temp`, `avg_humidity`
- [ ] Célula 2 — `city_ranking_temp` com coluna `temp_rank`
- [ ] Célula 3 — View `weather_alerts` criada com coluna `alert_type`
- [ ] Célula 4 — OPTIMIZE + ZORDER executados
- [ ] Validei as 3 tabelas/views no Data Explorer

In [0]:
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "workshop_weather", "Catálogo UC")
CATALOG = dbutils.widgets.get("catalog")

print(f"Catálogo : {CATALOG}")
print(f"Silver   : {CATALOG}.silver")
print(f"Gold     : {CATALOG}.gold")

---
## Célula 1 — `gold.daily_summary`

Agrega as previsões da Silver por cidade e por dia,  
calculando médias, extremos e a condição meteorológica dominante.

In [0]:
%sql
CREATE OR REPLACE TABLE workshop_weather.gold.daily_summary
USING DELTA
COMMENT 'Resumo diário por cidade: temperatura, umidade, vento e condição dominante'
AS
SELECT
    city_id,
    city_name,
    country,
    CAST(DATE_TRUNC('DAY', forecast_timestamp) AS DATE)  AS forecast_date,
    ROUND(AVG(temp), 2)                                  AS avg_temp,
    ROUND(MIN(temp), 2)                                  AS min_temp,
    ROUND(MAX(temp), 2)                                  AS max_temp,
    ROUND(AVG(humidity), 1)                              AS avg_humidity,
    ROUND(AVG(pop) * 100, 1)                             AS avg_rain_prob_pct,
    -- Condição dominante do dia (valor mais frequente)
    MODE(weather_main)                                   AS dominant_weather,
    COUNT(*)                                             AS slot_count,
    CURRENT_TIMESTAMP()                                  AS _updated_at
FROM workshop_weather.silver.forecast
WHERE _dq_status = 'valid'
GROUP BY city_id, city_name, country, DATE_TRUNC('DAY', forecast_timestamp)
ORDER BY city_name, forecast_date;

In [0]:
%sql
SELECT * FROM workshop_weather.gold.daily_summary
ORDER BY city_name, forecast_date
LIMIT 20;

---
## Célula 2 — `gold.city_ranking_temp`

Ranking de cidades pela temperatura **atual** (mais recente na Silver),  
usando `ROW_NUMBER()` window function particionada por país.

In [0]:
%sql
CREATE OR REPLACE TABLE workshop_weather.gold.city_ranking_temp
USING DELTA
COMMENT 'Ranking de cidades por temperatura atual, particionado por país'
AS
WITH latest AS (
    -- Pega apenas o registro mais recente por cidade
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY city_id
            ORDER BY weather_timestamp DESC
        ) AS rn
    FROM workshop_weather.silver.current_weather
    WHERE _dq_status = 'valid'
)
SELECT
    city_id,
    city_name,
    country,
    lat,
    lon,
    ROUND(temp, 1)        AS temp,
    ROUND(feels_like, 1)  AS feels_like,
    humidity,
    weather_main,
    weather_timestamp,
    -- Ranking dentro do país: 1 = cidade mais quente
    ROW_NUMBER() OVER (
        PARTITION BY country
        ORDER BY temp DESC
    )                     AS temp_rank,
    CURRENT_TIMESTAMP()   AS _updated_at
FROM latest
WHERE rn = 1;

In [0]:
%sql
SELECT city_name, country, temp, feels_like, weather_main, temp_rank
FROM workshop_weather.gold.city_ranking_temp
ORDER BY temp_rank;

---
## Célula 3 — View `gold.weather_alerts`

**View vs Table:** Esta é uma `VIEW` lógica — sempre reflete os dados atuais da Silver,  
sem armazenamento físico. Ideal para alertas em tempo real.

**Regras de alerta:**
- `HIGH_TEMP` → temperatura prevista > 35°C
- `RAIN_RISK` → probabilidade de precipitação > 70%
- `STORM_RISK` → condição `Thunderstorm` na previsão

In [0]:
%sql
CREATE OR REPLACE VIEW workshop_weather.gold.weather_alerts AS
SELECT
    city_id,
    city_name,
    country,
    forecast_timestamp,
    ROUND(temp, 1)           AS temp,
    ROUND(pop * 100, 0)      AS rain_prob_pct,
    weather_main,
    weather_description,
    -- Tipo de alerta: lógica de negócio centralizada aqui, não no dashboard
    CASE
        WHEN weather_main = 'Thunderstorm'  THEN 'STORM_RISK'
        WHEN temp > 35                      THEN 'HIGH_TEMP'
        WHEN pop > 0.70                     THEN 'RAIN_RISK'
        ELSE                                     'NORMAL'
    END                      AS alert_type,
    -- Nível de urgência para ordenação em dashboards
    CASE
        WHEN weather_main = 'Thunderstorm'  THEN 1
        WHEN temp > 35                      THEN 2
        WHEN pop > 0.70                     THEN 3
        ELSE                                     4
    END                      AS alert_priority
FROM workshop_weather.silver.forecast
WHERE _dq_status = 'valid'
  AND forecast_timestamp >= CURRENT_TIMESTAMP()  -- apenas previsões futuras
;

In [0]:
%sql
-- Resumo de alertas por cidade
SELECT
    city_name,
    alert_type,
    COUNT(*) AS ocorrencias,
    MIN(forecast_timestamp) AS primeiro_alerta
FROM workshop_weather.gold.weather_alerts
WHERE alert_type != 'NORMAL'
GROUP BY city_name, alert_type
ORDER BY city_name, alert_type;

---
## Célula 4 — `gold.temperature_moving_avg` (Desafio 3)

**Média móvel de temperatura por cidade** com window functions de 24h e 72h.

**Por que média móvel?**
- Suaviza flutuações pontuais (ex: rajada de calor momentânea)
- Revela tendências de aquecimento/resfriamento ao longo do tempo
- Janelas de 24h (1 dia) e 72h (3 dias) capturam padrões diários e de curto prazo

**Como funciona `RANGE BETWEEN INTERVAL ... PRECEDING`?**
- Diferente de `ROWS BETWEEN` (conta N linhas), `RANGE BETWEEN` com `INTERVAL` define uma janela **temporal**
- Para cada medição, calcula a média de todas as medições da mesma cidade dentro das últimas 24h/72h
- Ordenado por `weather_timestamp`, particionado por `city_id`

In [0]:
%sql
CREATE OR REPLACE TABLE workshop_weather.gold.temperature_moving_avg
USING DELTA
COMMENT 'Média móvel de temperatura por cidade: janelas de 24h e 72h via window functions'
AS
WITH deduped AS (
    -- Deduplica por (city_id, weather_timestamp), mantendo o registro mais recente
    SELECT
        city_id, city_name, country,
        weather_timestamp, temp, feels_like, humidity,
        weather_main, _ingestion_timestamp,
        ROW_NUMBER() OVER (
            PARTITION BY city_id, weather_timestamp
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM workshop_weather.silver.current_weather
    WHERE _dq_status = 'valid'
)
SELECT
    city_id,
    city_name,
    country,
    weather_timestamp,
    ROUND(temp, 2)        AS temp,
    ROUND(feels_like, 2)  AS feels_like,
    humidity,
    weather_main,
    -- Média móvel 24h: todas as medições nas últimas 24 horas
    ROUND(
        AVG(temp) OVER (
            PARTITION BY city_id
            ORDER BY weather_timestamp
            RANGE BETWEEN INTERVAL 24 HOURS PRECEDING AND CURRENT ROW
        ), 2
    ) AS temp_avg_24h,
    -- Média móvel 72h: todas as medições nas últimas 72 horas
    ROUND(
        AVG(temp) OVER (
            PARTITION BY city_id
            ORDER BY weather_timestamp
            RANGE BETWEEN INTERVAL 72 HOURS PRECEDING AND CURRENT ROW
        ), 2
    ) AS temp_avg_72h,
    -- Número de observações em cada janela (para auditoria)
    COUNT(*) OVER (
        PARTITION BY city_id
        ORDER BY weather_timestamp
        RANGE BETWEEN INTERVAL 24 HOURS PRECEDING AND CURRENT ROW
    ) AS obs_count_24h,
    COUNT(*) OVER (
        PARTITION BY city_id
        ORDER BY weather_timestamp
        RANGE BETWEEN INTERVAL 72 HOURS PRECEDING AND CURRENT ROW
    ) AS obs_count_72h,
    -- Delta entre temp atual e média 24h (indicador de tendência)
    ROUND(
        temp - AVG(temp) OVER (
            PARTITION BY city_id
            ORDER BY weather_timestamp
            RANGE BETWEEN INTERVAL 24 HOURS PRECEDING AND CURRENT ROW
        ), 2
    ) AS temp_delta_24h,
    current_timestamp() AS _updated_at
FROM deduped
WHERE rn = 1
ORDER BY city_id, weather_timestamp;

In [0]:
%sql
-- Amostra da tabela: mostra temp atual vs médias móveis por cidade
SELECT
    city_name,
    weather_timestamp,
    temp,
    temp_avg_24h,
    temp_avg_72h,
    obs_count_24h,
    obs_count_72h,
    temp_delta_24h
FROM workshop_weather.gold.temperature_moving_avg
ORDER BY city_name, weather_timestamp
LIMIT 20;

---
## Célula 5 — OPTIMIZE + ZORDER nas tabelas Gold

**Por que OPTIMIZE nas tabelas Gold?**
- Gold é a camada mais consultada (BI, dashboards, ML, Genie Spaces)
- `OPTIMIZE` compacta muitos arquivos pequenos em poucos arquivos grandes
- `ZORDER BY` co-localiza dados por colunas de filtro frequente → menos arquivos lidos por query

**Resultado esperado:** `DESCRIBE DETAIL` antes/depois mostra redução no número de arquivos.

In [0]:
%sql
-- Inspeciona antes do OPTIMIZE
DESCRIBE DETAIL workshop_weather.gold.daily_summary;

In [0]:
%sql
-- Compacta arquivos e ordena por cidade + data (colunas mais usadas em filtros)
OPTIMIZE workshop_weather.gold.daily_summary
ZORDER BY (city_id, forecast_date);

In [0]:
%sql
OPTIMIZE workshop_weather.gold.city_ranking_temp
ZORDER BY (country, temp_rank);

In [0]:
%sql
OPTIMIZE workshop_weather.gold.temperature_moving_avg
ZORDER BY (city_id, weather_timestamp);

---
## ✅ Checklist Final — Módulo 3 + Desafio 3 concluído?

- [ ] `daily_summary` tem linhas com `avg_temp`, `min_temp`, `max_temp` por cidade/dia
- [ ] `city_ranking_temp` tem coluna `temp_rank` (1 = mais quente no país)
- [ ] `weather_alerts` view retorna previsões com `alert_type`
- [ ] `temperature_moving_avg` tem `temp_avg_24h` e `temp_avg_72h` calculados por cidade
- [ ] `temp_delta_24h` mostra a diferença entre a temp atual e a média 24h
- [ ] OPTIMIZE executado em todas as tabelas Gold
- [ ] Genie Space configurada para consultas em linguagem natural
- [ ] Você entende quando usar TABLE vs VIEW vs MATERIALIZED VIEW

**Próximo: `06_delta_advanced_features.ipynb`**

---
*Workshop: Engenharia de Dados com Databricks & Delta Lake — OpenWeatherMap API*